# Metadata Tagging and Filtered Retrieval with LlamaIndex

**Document:** `pharma-blob-sample.pdf` (10 pages, 8 bundled supplier documents)
**Input:** the page-level metadata produced in the previous task (`page_metadata.json`)
**Stack:** LlamaIndex `PDFReader` + BGE-small embeddings (retrieval only; no LLM, no API key)
**Author:** Bashar

The previous task worked out which document and document type every page belongs to. This notebook attaches that metadata to every chunk in a LlamaIndex index and tests whether filtering on it actually improves retrieval.

Three ways of retrieving are compared on the same questions:

| Mode | How |
|---|---|
| No filter | Plain vector search over all pages |
| Filter after retrieval (course code) | Retrieve the top chunks, then drop the ones whose metadata doesn't match |
| Filter during retrieval | Pass `MetadataFilters` to the retriever, so only matching chunks are searched |

## Step 0: Install dependencies

In [ ]:
%pip install -q llama-index-core llama-index-readers-file llama-index-embeddings-huggingface

## Step 1: The PDF and the page metadata from the previous task

In [1]:
import os, json

PDF_PATH = os.environ.get("BLOB_PDF", "/content/pharma-blob-sample.pdf")
if not os.path.exists(PDF_PATH):
    import urllib.request
    urllib.request.urlretrieve(
        "https://drive.google.com/uc?export=download&id=1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw", PDF_PATH)

# Output of the page-splitting task (Approach C). Embedded here so the notebook runs on its own.
PAGE_METADATA = [
    {"page": 0, "doc_id": 0, "doc_type": "Cover Letter"},
    {"page": 1, "doc_id": 1, "doc_type": "Certificate of Quality"},
    {"page": 2, "doc_id": 2, "doc_type": "Certificate of Quality"},
    {"page": 3, "doc_id": 3, "doc_type": "Packaging Specification"},
    {"page": 4, "doc_id": 3, "doc_type": "Packaging Specification"},
    {"page": 5, "doc_id": 4, "doc_type": "BSE/TSE Declaration"},
    {"page": 6, "doc_id": 5, "doc_type": "Material Description"},
    {"page": 7, "doc_id": 6, "doc_type": "Supplier Qualification"},
    {"page": 8, "doc_id": 6, "doc_type": "Supplier Qualification"},
    {"page": 9, "doc_id": 7, "doc_type": "Chain of Custody"},
]
# The identifier the same task read from each document (lot / document / record number), keyed by doc_id
DOC_IDENTIFIERS = {0: "", 1: "18356721", 2: "15102934", 3: "PKG-SPEC-2023-0847",
                   4: "BSE-RA-2023-AKTA-001", 5: "29184612-AD", 6: "SUP-2019-0847", 7: ""}

## Step 2: Load one Document per page and tag it

As in the course, `PDFReader` returns one `Document` per page, and metadata is attached in a loop. Two differences from the course code:

- **The real classifier output is used.** The course's example `doc_type_array` does not match this PDF: it calls pages 1–2 a Cover Letter, but page 2 (index 1) is a Certificate of Quality. Tagging with hand-typed labels that don't match the file would make every filter return the wrong pages.
- **`doc_id` and `doc_identifier` are added** alongside `doc_type`. Two pages can have the same type but belong to different documents (the two certificates), and only these fields can tell them apart.

The course numbers pages from 1 (`page_number = i + 1`); `page_metadata.json` numbers them from 0. Both are kept so neither is confused for the other.

In [2]:
from llama_index.readers.file import PDFReader

pages = PDFReader().load_data(PDF_PATH)
assert len(pages) == len(PAGE_METADATA), "page count does not match the metadata"

documents = []
for doc, meta in zip(pages, PAGE_METADATA):
    doc.metadata = {
        "page_number": meta["page"] + 1,      # 1-based, as in the course
        "page_index": meta["page"],           # 0-based, as in page_metadata.json
        "source_file": os.path.basename(PDF_PATH),
        "doc_id": meta["doc_id"],
        "doc_type": meta["doc_type"],
        "doc_identifier": DOC_IDENTIFIERS[meta["doc_id"]],
    }
    # LlamaIndex embeds metadata together with the text by default. Keep the document type in the
    # embedding (it describes the content); keep bookkeeping fields out of it.
    doc.excluded_embed_metadata_keys = ["page_number", "page_index", "source_file", "doc_id", "doc_identifier"]
    documents.append(doc)

print(documents[1].metadata)

{'page_number': 2, 'page_index': 1, 'source_file': 'pharma-blob-sample.pdf', 'doc_id': 1, 'doc_type': 'Certificate of Quality', 'doc_identifier': '18356721'}


## Step 3: Chunk and index

Pages are split into 128-token chunks, the same size as in the earlier projects. **Metadata set on a `Document` is copied to every chunk made from it**, so filtering works at chunk level without extra code. `chunk_index`, the chunk's position within its page, is added afterwards because it only exists once the page has been split.

In [3]:
import pandas as pd
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = None
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    query_instruction="Represent this sentence for searching relevant passages: ")

nodes = SentenceSplitter(chunk_size=128, chunk_overlap=20).get_nodes_from_documents(documents)
position = {}
for node in nodes:
    page = node.metadata["page_number"]
    node.metadata["chunk_index"] = position.get(page, 0)
    position[page] = node.metadata["chunk_index"] + 1
    node.excluded_embed_metadata_keys = node.excluded_embed_metadata_keys + ["chunk_index"]

index = VectorStoreIndex(nodes)
print(f"{len(documents)} pages -> {len(nodes)} chunks indexed\n")
pd.DataFrame([{**{k: n.metadata[k] for k in ["page_number", "chunk_index", "doc_id", "doc_type", "doc_identifier"]},
               "text": n.get_content()[:50].replace("\n", " ")} for n in nodes])

LLM is explicitly disabled. Using MockLLM.


10 pages -> 44 chunks indexed



,page_number,chunk_index,doc_id,doc_type,doc_identifier,text
0,1,0,0,Cover Letter,,"Cytiva 100 Results Way Marlborough, MA 01752 U..."
1,1,1,0,Cover Letter,,The recommended storage temperature for standa...
2,1,2,0,Cover Letter,,"However, the operating temperature of AKTA rea..."
3,1,3,0,Cover Letter,,"Modified, AKTA ready 29184612 +2°C to +40°C Op..."
4,2,0,1,Certificate of Quality,18356721,Certificate of Quality This product is manufac...
5,2,1,1,Certificate of Quality,18356721,Product: AKTA ready Low Flow Kit Lot Number: 1...
6,2,2,1,Certificate of Quality,18356721,AKTA ready Expiration Date: 20260315 Product R...
7,2,3,1,Certificate of Quality,18356721,Description Test Method Result Autoclave - Pum...
8,3,0,2,Certificate of Quality,15102934,Certificate of Quality This product is manufac...
9,3,1,2,Certificate of Quality,15102934,Product: AKTA ready High Flow Kit Lot Number: ...


## Step 4: Questions and what a correct retrieval looks like

Each question names the document that holds the answer, so a retrieved chunk is **on target** if it comes from that document (by `doc_id`).

- **Q1, Q2** are the course's own examples.
- **Q3** asks about one of the two near-identical certificates.
- **Q4** is the scenario from the project overview: a question about the packaging specification, which a plain search answers from the cover letter instead. (The packaging specification lists no storage temperature at all, so the correct outcome is to retrieve the packaging specification and find the answer is not there, not to return the cover letter's "+5 °C".)

`top_k=2` is used for every mode because it is LlamaIndex's default, and the course code calls `index.as_retriever()` without changing it.

In [4]:
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter

TOP_K = 2
QUESTIONS = [
    {"id": "Q1", "query": "What are the BSE/TSE compliance dates for this product?",
     "filters": {"doc_type": "BSE/TSE Declaration"}, "target_doc": 4},
    {"id": "Q2", "query": "Give me the part numbers.",
     "filters": {"doc_type": "Material Description"}, "target_doc": 5},
    {"id": "Q3", "query": "What is the expiration date on the certificate for the High Flow Kit, lot 15102934?",
     "filters": {"doc_type": "Certificate of Quality", "doc_identifier": "15102934"}, "target_doc": 2},
    {"id": "Q4", "query": "What's the storage temperature listed in the packaging spec?",
     "filters": {"doc_type": "Packaging Specification"}, "target_doc": 3},
]

def no_filter(q):
    return index.as_retriever(similarity_top_k=TOP_K).retrieve(q["query"])

def filter_after(q):
    # The course's approach: retrieve first, then keep only chunks whose metadata matches
    results = index.as_retriever(similarity_top_k=TOP_K).retrieve(q["query"])
    return [r for r in results if all(r.metadata.get(k) == v for k, v in q["filters"].items())]

def filter_during(q):
    # Filters are applied inside the vector search, so the top-k is chosen from matching chunks only
    filters = MetadataFilters(filters=[MetadataFilter(key=k, value=v) for k, v in q["filters"].items()])
    return index.as_retriever(similarity_top_k=TOP_K, filters=filters).retrieve(q["query"])

MODES = {"No filter": no_filter, "Filter after retrieval": filter_after, "Filter during retrieval": filter_during}

## Step 5: Run every question through every mode

In [5]:
rows = []
for q in QUESTIONS:
    print(f"\n{'=' * 100}\n{q['id']}: {q['query']}\n   filters: {q['filters']}   (answer is in doc {q['target_doc']})\n{'=' * 100}")
    for mode, retrieve in MODES.items():
        results = retrieve(q)
        on_target = sum(r.metadata["doc_id"] == q["target_doc"] for r in results)
        rows.append({"Question": q["id"], "Mode": mode, "Chunks returned": len(results),
                     "On target": f"{on_target}/{len(results)}" if results else "0/0 (empty)",
                     "Docs returned": sorted({r.metadata['doc_type'] + f' (doc {r.metadata["doc_id"]})' for r in results})})
        print(f"\n  [{mode}] {len(results)} chunk(s)")
        for r in results:
            mark = "on target" if r.metadata["doc_id"] == q["target_doc"] else "OFF TARGET"
            print(f"    score={r.score:.3f} | page {r.metadata['page_number']} | {r.metadata['doc_type']} (doc {r.metadata['doc_id']}) "
                  f"| {mark}\n      {r.get_content()[:150].replace(chr(10), ' | ')}")
        if not results:
            print("    (nothing returned)")


Q1: What are the BSE/TSE compliance dates for this product?
   filters: {'doc_type': 'BSE/TSE Declaration'}   (answer is in doc 4)

  [No filter] 2 chunk(s)
    score=0.772 | page 6 | BSE/TSE Declaration (doc 4) | on target
      (animal by-products) | - United States Pharmacopeia (USP) <87> and <88> | Risk Assessment Date: 15 March 2023 | Assessment Valid Until: 15 March 2028 | Assess
    score=0.770 | page 6 | BSE/TSE Declaration (doc 4) | on target
      declaration is based on information provided by our raw material suppliers and our own assessment of the | manufacturing process. | Signed: Quality Assura

  [Filter after retrieval] 2 chunk(s)
    score=0.772 | page 6 | BSE/TSE Declaration (doc 4) | on target
      (animal by-products) | - United States Pharmacopeia (USP) <87> and <88> | Risk Assessment Date: 15 March 2023 | Assessment Valid Until: 15 March 2028 | Assess
    score=0.770 | page 6 | BSE/TSE Declaration (doc 4) | on target
      declaration is based on information pr


  [Filter during retrieval] 2 chunk(s)
    score=0.683 | page 5 | Packaging Specification (doc 3) | on target
      Packaging Component Specification (continued) | Document Number: PKG-SPEC-2023-0847, Page 2 of 2 | 4. Configuration Change History | Rev Date ECN Number Cha
    score=0.675 | page 4 | Packaging Specification (doc 3) | on target
      Packaging Component Specification | Document Number: PKG-SPEC-2023-0847 | Revision: C | Effective Date: 2023-11-08 | Component: AKTA ready Flow Kit Packaging 


## Step 6: Summary

In [6]:
summary = pd.DataFrame(rows)
summary.pivot(index="Question", columns="Mode", values="On target")[list(MODES)]

Mode,No filter,Filter after retrieval,Filter during retrieval
Question,,,
Q1,2/2,2/2,2/2
Q2,0/2,0/0 (empty),2/2
Q3,1/2,1/1,2/2
Q4,0/2,0/0 (empty),2/2


## Step 7: Metadata for traceability

The bonus in the module: every retrieved chunk carries its metadata, so an answer can be shown with exactly where it came from. These are the chunks behind Q1 with the filter applied during retrieval.

In [7]:
for r in filter_during(QUESTIONS[0]):
    print({k: r.metadata[k] for k in ["source_file", "page_number", "doc_id", "doc_type", "doc_identifier", "chunk_index"]})

{'source_file': 'pharma-blob-sample.pdf', 'page_number': 6, 'doc_id': 4, 'doc_type': 'BSE/TSE Declaration', 'doc_identifier': 'BSE-RA-2023-AKTA-001', 'chunk_index': 3}
{'source_file': 'pharma-blob-sample.pdf', 'page_number': 6, 'doc_id': 4, 'doc_type': 'BSE/TSE Declaration', 'doc_identifier': 'BSE-RA-2023-AKTA-001', 'chunk_index': 4}


## Findings

Chunks from the document that holds the answer, out of chunks returned (`top_k=2`, LlamaIndex's default):

| Question | No filter | Filter after retrieval (course code) | Filter during retrieval |
|---|---|---|---|
| Q1 BSE/TSE compliance dates | 2/2 | 2/2 | 2/2 |
| Q2 Part numbers, Material Description only | 0/2 | **0, nothing returned** | 2/2 |
| Q3 Expiry date, High Flow Kit certificate | 1/2 | 1/1 | 2/2 |
| Q4 Storage temperature "in the packaging spec" | 0/2 | **0, nothing returned** | 2/2 |

**1. Filtering during retrieval was the only mode that always searched the right document.** It returned chunks only from the requested document for every question. When the question itself names a common topic, a plain search often does not: "part numbers" appear in five of the eight documents, and the plain search picked the Supplier Qualification Record and the Packaging Specification.

**2. The course's filter-after-retrieval approach can return nothing.** It keeps the top 2 chunks and *then* discards those of the wrong type. If neither of the top 2 is from the requested document, nothing is left: that happened for Q2 and Q4. Filtering during retrieval chooses the top 2 *from* the matching chunks, so it always returns something when the document exists. (The course's recap also suggests `filters={"doc_type": ...}`; the working LlamaIndex form is `MetadataFilters(filters=[MetadataFilter(key="doc_type", value=...)])`.)

**3. The project overview's failure case is real, and the filter fixes it.** Q4 asks for the storage temperature "listed in the packaging spec". The plain search returned the cover letter's "+5 °C" and the Material Description's operating range. Those are confident-looking but wrong sources, because the packaging specification lists no storage temperature at all. With the filter, only packaging-specification chunks are retrieved, so an LLM would correctly report that the spec does not state one.

**4. `doc_type` alone cannot pick between two certificates; `doc_identifier` can.** For Q3 the plain search returned one chunk from each Certificate of Quality, and the two look almost identical. Filtering on both `doc_type` and `doc_identifier` (lot 15102934) returned only the High Flow Kit certificate, including its expiration date (20210712). This is why the page-splitting step records an identifier and not only a type.

**5. A filter picks the right document, not the best chunk within it.** For Q2, the filtered results came from the Material Description, but the chunk with "Part Number: 29184612" ranked 3rd within that document, just outside `top_k=2`. The filter fixes *where* to search; `top_k` and ranking still decide *what* comes back.

**6. Metadata is not free.** LlamaIndex counts the metadata text against the chunk size when splitting, so about 130 characters of tags per chunk leave less room for the page's own text: the 10 pages became 44 chunks. Bookkeeping fields such as file name, page number and IDs were kept out of the embedding so they do not influence similarity; `doc_type` was kept in because it describes the content.

**Limit:** four questions on one 10-page bundle. In each case the filter to use was written by hand. Choosing the filter automatically from the user's question is the routing problem in the next step.